## **Analyse Sémantique des Données Textuelles : Projet final**

Objectif du projet : Proposer une analyse sémantique d'une oeuvre en anglais & libre de droit. L'oeuvre choisie pour ce projet est : ***L'art de la Guerre*** de *Sun Tzu* soit ***The Art of War*** en anglais.

### ***Téléchargement du livre***

Grâce à la bibliothèque requests, le livre sera directement téléchargé depuis le site ***Project Gutemberg***

In [1]:
import requests

# on crée une fonction qui ira télécharger le livre directement sur le site internet
url = "https://www.gutenberg.org/cache/epub/132/pg132.txt"
def get_book(url):
    art_of_war = "art_of_war.txt"
    
    try:
        response = requests.get(url)
        response.raise_for_status()
        
        with open(art_of_war, "w",encoding="utf-8") as file:
            file.write(response.text)
        print("Le livre 'L'Art de la Guerre' a bien été récupéré.")
    except requests.exceptions.RequestException as exception:
        print(f"Le livre n'a pas été téléchargé car {exception}")
        
get_book("https://www.gutenberg.org/cache/epub/132/pg132.txt")

Le livre 'L'Art de la Guerre' a bien été récupéré.


Après avoir récupéré le livre, il faut enlever tout le texte qui n'est pas nécéssaire pour l'analyse i.e. le texte avant la mention ***START OF THE PROJECT...*** et après la mention ***END OF THE PROJECT...***

In [2]:
# On importe la bibliothèque native re

import re

# Charger le texte brut
with open("art_of_war.txt", "r", encoding="utf-8") as f:
    text = f.read()

# Regex pour enlever tout avant le START et après le END
pattern = r"\*\*\* START OF THE PROJECT GUTENBERG EBOOK.*?\*\*\*(.*?)\*\*\* END OF THE PROJECT GUTENBERG EBOOK"

# re.DOTALL pour que '.' capture aussi les sauts de ligne
match = re.search(pattern, text, re.DOTALL | re.IGNORECASE)
if match:
    cleaned_text = match.group(1).strip()  # le contenu réel
else:
    cleaned_text = text.strip()  # au cas où les marqueurs sont absents

# Vérification rapide
print(cleaned_text[:500])

# Sauvegarder la version nettoyée
with open("art_of_war_clean.txt", "w", encoding="utf-8") as f:
    f.write(cleaned_text)

Sun Tzŭ
on
The Art of War

THE OLDEST MILITARY TREATISE IN THE WORLD
Translated from the Chinese with Introduction and Critical Notes

BY
LIONEL GILES, M.A.

Assistant in the Department of Oriental Printed Books and MSS.
in the British Museum




1910



To my brother
Captain Valentine Giles, R.G.
in the hope that
a work 2400 years old
may yet contain lessons worth consideration
by the soldier of today
this translation
is affectionately dedicated.



Contents


 Preface to the Project Gutenberg 


Après avoir nettoyé le fichier initial pour n'en garder que le contenu, il faudra pour l'analyse le tokeniser en deux étapes : La première étape consistera à le tokeniser en phrase i.e. découper le texte en liste de phrases puis découper chaque phrase en mot.

Le texte est tokenizé en phrases : 

In [3]:
import nltk
from nltk.tokenize import sent_tokenize

# Pour la ponctuation, il faut télécharger 'punkt' de la librairie nltk
nltk.download('punkt')

#Il faut ensuite ouvrir le fichier :
aow_raw = open('art_of_war_clean.txt').read()
aow_sentenced_tokenized = sent_tokenize(aow_raw)

[nltk_data] Downloading package punkt to /Users/loram/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [4]:
len(aow_sentenced_tokenized)

3194

L'Art de La Guerre contient donc 3194 phrases. Voici les 500 premières phrases du livre : 

In [5]:
aow_sentenced_tokenized[:500]

['Sun Tzŭ\non\nThe Art of War\n\nTHE OLDEST MILITARY TREATISE IN THE WORLD\nTranslated from the Chinese with Introduction and Critical Notes\n\nBY\nLIONEL GILES, M.A.',
 'Assistant in the Department of Oriental Printed Books and MSS.',
 'in the British Museum\n\n\n\n\n1910\n\n\n\nTo my brother\nCaptain Valentine Giles, R.G.',
 'in the hope that\na work 2400 years old\nmay yet contain lessons worth consideration\nby the soldier of today\nthis translation\nis affectionately dedicated.',
 'Contents\n\n\n Preface to the Project Gutenberg Etext\n Preface by Lionel Giles\n INTRODUCTION\n Sun Wu and his Book\n The Text of Sun Tzŭ\n The Commentators\n Appreciations of Sun Tzŭ\n Apologies for War\n Bibliography\n Chapter I.',
 'Laying plans\n Chapter II.',
 'Waging War\n Chapter III.',
 'Attack by Stratagem\n Chapter IV.',
 'Tactical Dispositions\n Chapter V. Energy\n Chapter VI.',
 'Weak Points and Strong\n Chapter VII Manœuvring\n Chapter VIII.',
 'Variation of Tactics\n Chapter IX.',
 'The A

Malgré le premier filtrage, il semble qu'une partie du texte ne soit pas directement des écrits de Sun Tzu mais une liste de chapitres, suivi d'une préface et une introduction. Aucun de ces éléments n'a été écrit par Sun Tzu. Étant donné le décalage d'époque entre Sun Tzu et Lionel Giles, pour maximiser l'analyse et éviter de polluer le texte avec des écrits qui seraient logiquement d'un point de vue syntaxique éloigné du style de Sun Tzu, un second filtrage sera réalisé. On gardera la liste des chapitres dans une autre variable et on refiltrera pour ne garder uniquement ce qui a été écrit par Sun Tzu :

In [6]:
''' 
Avec la méthode search re du module re, 
on cherche la position de la première apparition du mot
Contents et de Preface by. Cette recherche permettra d'isoler
la liste de chapitre pour la stocker dans une variable
'''
motif_contents = r"Contents"
match_contents = re.search(motif_contents,aow_raw,re.DOTALL)

'''
Pour être sûr de bien capturer la liste des chapitre et dû
au fait qu'il y a plusieurs fois le mot Preface by dans le début
du texte, on commence la recherche de la première occurence
à partir de l'index où se termine le mot "Contents cherché grâce
à match_contents et le +100 indique à Python que la recherche doit
commencer 100 caractères plus loin après la fin de "Contents"
'''
pattern = re.compile(r"Preface.*?Etext", re.DOTALL)
match_preface = pattern.search(aow_raw, pos=match_contents.end()+ 100)
''' 
Si re trouve une correspondance, 
on stocke la position dans une variable puis après on
découpera le texte pour stocker la liste des chapitres
'''
if match_contents and match_preface:
    aow_chapter_list = aow_raw[match_contents.start():match_preface.start()].strip()
print(len(aow_chapter_list))

594


In [7]:
print(aow_chapter_list)

Contents


 Preface to the Project Gutenberg Etext
 Preface by Lionel Giles
 INTRODUCTION
 Sun Wu and his Book
 The Text of Sun Tzŭ
 The Commentators
 Appreciations of Sun Tzŭ
 Apologies for War
 Bibliography
 Chapter I. Laying plans
 Chapter II. Waging War
 Chapter III. Attack by Stratagem
 Chapter IV. Tactical Dispositions
 Chapter V. Energy
 Chapter VI. Weak Points and Strong
 Chapter VII Manœuvring
 Chapter VIII. Variation of Tactics
 Chapter IX. The Army on the March
 Chapter X. Terrain
 Chapter XI. The Nine Situations
 Chapter XII. The Attack by Fire
 Chapter XIII. The Use of Spies


En reproduisant la stratégie utilisée pour isoler la liste des chapitres on va isoler le texte écrit uniquement par Sun Tzu :

In [8]:
''' On ajoute un \\ entrer I et . pour spécifier à Python de chercher 
directement le caractère . 
re.IGNORECASE permet de spécifer au moteur de recherche d'ignorer les
différences entre miniscules et majuscules. '''
pattern_beginning_aow = re.compile(r"Chapter I\. Laying plans",re.IGNORECASE)
chapter_i_match = pattern_beginning_aow.search(aow_raw, pos=match_preface.end())

if chapter_i_match:
    aow_final_version = aow_raw[chapter_i_match.start():].strip()
else:
    print("Le début du chapitre 1 n'a pas été détecté par re.")


Le code de détection du début du premier chapitre a bien fonctionné il faut vérifier que le début du texte a bien été détecté en s'assurant que chapter_i_match retourne bien quelque chose :

In [9]:
chapter_i_match

<re.Match object; span=(81558, 81581), match='Chapter I. LAYING PLANS'>

On affiche le type de aow_final_version qui sera la variable sur laquelle on travaille à partir de maintenant ainsi que les 500 premiers caractères du texte :

In [10]:
print(aow_final_version.__class__,aow_final_version[:500])

<class 'str'> Chapter I. LAYING PLANS

[Ts’ao Kung, in defining the meaning of the Chinese for the title of
this chapter, says it refers to the deliberations in the temple
selected by the general for his temporary use, or as we should say, in
his tent. See. § 26.]


1. Sun Tzŭ said: The art of war is of vital importance to the State.

2. It is a matter of life and death, a road either to safety or to
ruin. Hence it is a subject of inquiry which can on no account be
neglected.

3. The art of war, then, is gove


On sauvegarde le texte final dans un nouveau fichier txt :

In [11]:
with open("art_of_war_txt_only.txt","w") as f:
    f.write(aow_final_version)
print("Le texte a été sauvegardé dans art_of_war_txt_only.txt")

Le texte a été sauvegardé dans art_of_war_txt_only.txt


### **II. Tokenisation**

Après avoir conservé uniquement le texte final sans les préfaces ou introductions, la seconde étape du projet est la ***tokenisation*** du texte. Cette tokenisation sera réalisée en deux étapes : 
- d'abord on tokenisera le texte en phrases.
- ensuite, on tokenisera les phrases en mots.

In [12]:
# Il faut tout d'abord télécharger punkt pour la segmentation textuelle.
import nltk
nltk.download('punkt')
# Pour pouvoir appliquer les règles de segmentations dans des langues comme l'anglais, Punkt a besoin de charger les tables de données se trouvant sur punkt_tab
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /Users/loram/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /Users/loram/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

#### **Tokenisation du texte en phrase :**

Cette partie nous permettra de parcourir l'intégralité de notre texte et retournera dans ***sentences_aow*** une liste ou chaque élément de la liste est une phrase :

In [13]:
from nltk.tokenize import sent_tokenize

sentences_aow = sent_tokenize(aow_final_version)
print("Le texte a bien été tokenizé en phrase.")

Le texte a bien été tokenizé en phrase.


Vérifions le type de *sentences_aow* ainsi que les 3 premières phrases :

In [14]:
print(sentences_aow.__class__,sep= "\n")
limit = 11
for i, sentence in enumerate(sentences_aow[:10], 1):
    print(f"phrase n°{i} : {sentence}")

<class 'list'>
phrase n°1 : Chapter I.
phrase n°2 : LAYING PLANS

[Ts’ao Kung, in defining the meaning of the Chinese for the title of
this chapter, says it refers to the deliberations in the temple
selected by the general for his temporary use, or as we should say, in
his tent.
phrase n°3 : See.
phrase n°4 : § 26.]
phrase n°5 : 1.
phrase n°6 : Sun Tzŭ said: The art of war is of vital importance to the State.
phrase n°7 : 2.
phrase n°8 : It is a matter of life and death, a road either to safety or to
ruin.
phrase n°9 : Hence it is a subject of inquiry which can on no account be
neglected.
phrase n°10 : 3.


Ce qui ressort de cette tokenization est que NLTK a très bien détecté les débuts et fin de phrases mais ne comprend pas qu'un numéro suivi d'un point puis d'une phrase sont une seule et même phrase. Pour tenter de régler ce problème, on essaiera de passer par l'expression régulière pour supprimer tous les points après les chiffres se situant en début de phrase :

In [15]:
import re
# ^ permet de cibler le début de ligne
# (\d+) capture le nombre, \. cible le point littéral
# \g<1> reinjecte le nombre capturer dans le texte de remplacement
# re.MULTILINE permet à Python de regarder ce qui se trouve après chaque saut de ligne

aow_final_version = re.sub(r"^(\d+)\.", r"\g<1>", aow_final_version, flags=re.MULTILINE)
sentences_aow = sent_tokenize(aow_final_version)
print("Le texte a bien été tokenizé en phrases.")

Le texte a bien été tokenizé en phrases.


Vérifions si cette fois, le problème a été résolu :

In [16]:
for i, sentence in enumerate(sentences_aow[:10],1):
    print(f"phrase n°{i} : {sentence}")

phrase n°1 : Chapter I.
phrase n°2 : LAYING PLANS

[Ts’ao Kung, in defining the meaning of the Chinese for the title of
this chapter, says it refers to the deliberations in the temple
selected by the general for his temporary use, or as we should say, in
his tent.
phrase n°3 : See.
phrase n°4 : § 26.]
phrase n°5 : 1 Sun Tzŭ said: The art of war is of vital importance to the State.
phrase n°6 : 2 It is a matter of life and death, a road either to safety or to
ruin.
phrase n°7 : Hence it is a subject of inquiry which can on no account be
neglected.
phrase n°8 : 3 The art of war, then, is governed by five constant factors, to be
taken into account in one’s deliberations, when seeking to determine
the conditions obtaining in the field.
phrase n°9 : 4 These are: (1) The Moral Law; (2) Heaven; (3) Earth; (4) The
Commander; (5) Method and discipline.
phrase n°10 : [It appears from what follows that Sun Tzŭ means by "Moral Law" a
principle of harmony, not unlike the Tao of Lao Tzŭ in its moral

Les points ont disparu après les chiffres mais NLTK a maintenant bien découpé les phrases.

In [17]:
print(f"L'Art de la Guerre contient {len(sentences_aow)} phrases.")

L'Art de la Guerre contient 1942 phrases.


Après avoir tokenizé le texte en phrases, il faudra le tokenizer en mot :

In [23]:
# Il faut impérativement relancer cette étape
words_aow = [nltk.word_tokenize(sentence) for sentence in sentences_aow]

Après avoir tokenizé le texte en mots, il faut retirer tous les stopwords : 

In [24]:
from nltk.corpus import stopwords

stopwords_set = set(stopwords.words('english'))

''' On parcours words_aow qui est la liste de listes 
dont les listes sont des phrases découpés en mots pour parcourir les mots du texte.
Ensuite, on met le mot en miniscules, puis on le garde si c'est un chiffre ou un mot et 
on vérifie que le mot n'est pas un stopword.'''

cleaned_sentences_aow = [[w.lower() for w in sentence if w.isalnum() and w.lower() not in stopwords_set] for sentence in words_aow]

print(cleaned_sentences_aow[:50])

[['chapter'], ['laying', 'plans', 'ts', 'ao', 'kung', 'defining', 'meaning', 'chinese', 'title', 'chapter', 'says', 'refers', 'deliberations', 'temple', 'selected', 'general', 'temporary', 'use', 'say', 'tent'], ['see'], ['26'], ['1', 'sun', 'tzŭ', 'said', 'art', 'war', 'vital', 'importance', 'state'], ['2', 'matter', 'life', 'death', 'road', 'either', 'safety', 'ruin'], ['hence', 'subject', 'inquiry', 'account', 'neglected'], ['3', 'art', 'war', 'governed', 'five', 'constant', 'factors', 'taken', 'account', 'one', 'deliberations', 'seeking', 'determine', 'conditions', 'obtaining', 'field'], ['4', '1', 'moral', 'law', '2', 'heaven', '3', 'earth', '4', 'commander', '5', 'method', 'discipline'], ['appears', 'follows', 'sun', 'tzŭ', 'means', 'moral', 'law', 'principle', 'harmony', 'unlike', 'tao', 'lao', 'tzŭ', 'moral', 'aspect'], ['one', 'might', 'tempted', 'render', 'morale', 'considered', 'attribute', '13'], ['5', '6'], ['moral', 'causes', 'people', 'complete', 'accord', 'ruler', 'foll

In [25]:
type(words_aow[0])

list

In [26]:
cleaned_sentences_aow

[['chapter'],
 ['laying',
  'plans',
  'ts',
  'ao',
  'kung',
  'defining',
  'meaning',
  'chinese',
  'title',
  'chapter',
  'says',
  'refers',
  'deliberations',
  'temple',
  'selected',
  'general',
  'temporary',
  'use',
  'say',
  'tent'],
 ['see'],
 ['26'],
 ['1', 'sun', 'tzŭ', 'said', 'art', 'war', 'vital', 'importance', 'state'],
 ['2', 'matter', 'life', 'death', 'road', 'either', 'safety', 'ruin'],
 ['hence', 'subject', 'inquiry', 'account', 'neglected'],
 ['3',
  'art',
  'war',
  'governed',
  'five',
  'constant',
  'factors',
  'taken',
  'account',
  'one',
  'deliberations',
  'seeking',
  'determine',
  'conditions',
  'obtaining',
  'field'],
 ['4',
  '1',
  'moral',
  'law',
  '2',
  'heaven',
  '3',
  'earth',
  '4',
  'commander',
  '5',
  'method',
  'discipline'],
 ['appears',
  'follows',
  'sun',
  'tzŭ',
  'means',
  'moral',
  'law',
  'principle',
  'harmony',
  'unlike',
  'tao',
  'lao',
  'tzŭ',
  'moral',
  'aspect'],
 ['one',
  'might',
  'tempted'

### **Étude du contenu du texte, chapitre par chapitre**

Après avoir tokenizé le texte en phrases puis en mots, on va maintenant appliquer TF-IDF sur notre texte sur les 3 premiers chapitres du livre :

In [41]:
print(aow_chapter_list.__class__,aow_chapter_list)

<class 'str'> Contents


 Preface to the Project Gutenberg Etext
 Preface by Lionel Giles
 INTRODUCTION
 Sun Wu and his Book
 The Text of Sun Tzŭ
 The Commentators
 Appreciations of Sun Tzŭ
 Apologies for War
 Bibliography
 Chapter I. Laying plans
 Chapter II. Waging War
 Chapter III. Attack by Stratagem
 Chapter IV. Tactical Dispositions
 Chapter V. Energy
 Chapter VI. Weak Points and Strong
 Chapter VII Manœuvring
 Chapter VIII. Variation of Tactics
 Chapter IX. The Army on the March
 Chapter X. Terrain
 Chapter XI. The Nine Situations
 Chapter XII. The Attack by Fire
 Chapter XIII. The Use of Spies


In [46]:
''' On part de la liste des chapitres contenus dans aow_chapter_list 
pour ne garder que les chapitres. aow_chapter_list est de type string,
il faut le convertir en liste pour pouvoir ensuite découper ce texte chapitre
par chapitre'''

liste_chapitres = aow_chapter_list.splitlines()
liste_chapitres = [ l.strip() for l in liste_chapitres if re.search(r"Chapter\s+[IVXLC]+", l)
]
print(len(liste_chapitres),liste_chapitres)

13 ['Chapter I. Laying plans', 'Chapter II. Waging War', 'Chapter III. Attack by Stratagem', 'Chapter IV. Tactical Dispositions', 'Chapter V. Energy', 'Chapter VI. Weak Points and Strong', 'Chapter VII Manœuvring', 'Chapter VIII. Variation of Tactics', 'Chapter IX. The Army on the March', 'Chapter X. Terrain', 'Chapter XI. The Nine Situations', 'Chapter XII. The Attack by Fire', 'Chapter XIII. The Use of Spies']


Après avoir isolé la liste des chapitres, on va redécouper le texte

In [62]:
import re

# On cherche toutes les positions où commence le mot "Chapter" suivi d'un chiffre romain
# Le pattern r"Chapter\s+[IVXLC]+" est très robuste
balises = [match.start() for match in re.finditer(r"Chapter\s+[IVXLC]+", aow_final_version)]

# On ajoute la fin du texte pour fermer le dernier chapitre
balises.append(len(aow_final_version))

print(f"Nombre de balises trouvées : {len(balises) - 1}") 
# Si ça affiche 13, on a gagné.

Nombre de balises trouvées : 14


In [63]:


# Liste pour stocker le texte propre de chaque chapitre
corpus_aow_nettoye = []

for i in range(len(balises) - 1):
    # 1. Extraction du segment brut
    debut = balises[i]
    fin = balises[i+1]
    segment_brut = aow_final_version[debut:fin].strip()
    
    # 2. Nettoyage NLTK (Tokenisation + Stopwords + Alphanumérique)
    tokens = nltk.word_tokenize(segment_brut.lower())
    mots_propres = [w for w in tokens if w.isalnum() and w not in stopwords_set]
    
    # 3. On rejoint les mots en une seule chaîne pour le vectoriseur
    corpus_aow_nettoye.append(" ".join(mots_propres))

print(f"Corpus prêt : {len(corpus_aow_nettoye)} documents.")

Corpus prêt : 14 documents.


In [64]:
corpus_aow_nettoye

['chapter laying plans ts ao kung defining meaning chinese title chapter says refers deliberations temple selected general temporary use say tent see 26 1 sun tzŭ said art war vital importance state 2 matter life death road either safety ruin hence subject inquiry account neglected 3 art war governed five constant factors taken account one deliberations seeking determine conditions obtaining field 4 1 moral law 2 heaven 3 earth 4 commander 5 method discipline appears follows sun tzŭ means moral law principle harmony unlike tao lao tzŭ moral aspect one might tempted render morale considered attribute 13 5 6 moral causes people complete accord ruler follow regardless lives undismayed danger tu yu quotes wang tzŭ saying without constant practice officers nervous undecided mustering battle without constant practice general wavering irresolute crisis hand 7 signifies night day cold heat times seasons commentators think make unnecessary mystery two words meng shih refers hard soft waxing wan

Maintenant que les documents pour les 13 chapitres sont prêts, on peut commencer la création de la Matrice TF-IDF pour le premier chapitre :

In [65]:
corpus_par_chapitre = []

for i in range(len(balises) - 1):
    # 1. On extrait le texte brut du chapitre
    texte_chapitre = aow_final_version[balises[i]:balises[i+1]].strip()
    
    # 2. On le nettoie (on réutilise ta logique NLTK)
    tokens = nltk.word_tokenize(texte_chapitre.lower())
    mots_propres = [w for w in tokens if w.isalnum() and w not in stopwords_set]
    
    # 3. On ajoute la liste de mots au corpus
    corpus_par_chapitre.append(mots_propres)

print(f"Nombre de chapitres prêts pour le TF-IDF : {len(corpus_par_chapitre)}")

Nombre de chapitres prêts pour le TF-IDF : 14


In [66]:
corpus_par_chapitre = []

for i in range(len(balises) - 1):
    # 1. On extrait le texte brut du chapitre
    texte_chapitre = aow_final_version[balises[i]:balises[i+1]].strip()
    
    # 2. On le nettoie (on réutilise ta logique NLTK)
    tokens = nltk.word_tokenize(texte_chapitre.lower())
    mots_propres = [w for w in tokens if w.isalnum() and w not in stopwords_set]
    
    # 3. On ajoute la liste de mots au corpus
    corpus_par_chapitre.append(mots_propres)

print(f"Nombre de chapitres prêts pour le TF-IDF : {len(corpus_par_chapitre)}")

Nombre de chapitres prêts pour le TF-IDF : 14


In [67]:
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd

# On transforme les listes de mots en chaînes pour le vectoriseur
documents_pour_tfidf = [" ".join(mots) for mots in corpus_par_chapitre]

vectoriseur = TfidfVectorizer()
matrice_strategie = vectoriseur.fit_transform(documents_pour_tfidf)

# Tableau de résultats
tableau_tfidf = pd.DataFrame(
    matrice_strategie.toarray(),
    index=[f"Chapitre {i+1}" for i in range(len(documents_pour_tfidf))],
    columns=vectoriseur.get_feature_names_out()
)

# On vérifie les mots les plus importants du chapitre 1
print(tableau_tfidf.iloc[0].sort_values(ascending=False).head(10))

constant         0.159307
moral            0.159307
general          0.146697
law              0.143237
practice         0.143237
tzŭ              0.132028
temple           0.124120
calculations     0.120909
must             0.109807
deliberations    0.107428
Name: Chapitre 1, dtype: float64


In [68]:
# Extraction des mots-clés pour le Chapitre 2
print("Top 10 - Chapitre 2 (Waging War) :")
print(tableau_tfidf.iloc[1].sort_values(ascending=False).head(10))

print("\n" + "-"*30 + "\n")

# Extraction des mots-clés pour le Chapitre 3 (Attack by Stratagem)
print("Top 10 - Chapitre 3 (Attack by Stratagem) :")
print(tableau_tfidf.iloc[2].sort_values(ascending=False).head(10))

Top 10 - Chapitre 2 (Waging War) :
chariots    0.263367
army        0.188628
people      0.164214
war         0.148207
may         0.134734
heavy       0.132253
haste       0.131556
stupid      0.131556
enemy       0.121261
prices      0.113998
Name: Chapitre 2, dtype: float64

------------------------------

Top 10 - Chapitre 3 (Attack by Stratagem) :
army          0.263041
enemy         0.241121
man           0.144098
one           0.142480
general       0.109600
ch            0.105479
attack        0.098640
without       0.098640
equivalent    0.095217
destroy       0.095217
Name: Chapitre 3, dtype: float64


Phrase 1 (Chapitre 2) : La matrice TF-IDF du Chapitre 2 met en exergue la dimension logistique et matérielle de la guerre, où le terme "chariots" (0.26) domine le vecteur, illustrant une approche centrée sur l'infrastructure et les coûts économiques prohibitifs de la mobilisation.

Phrase 2 (Chapitre 3) : Dans le Chapitre 3, les coefficients traduisent une bascule vers la psychologie du combat, marquée par une forte corrélation sémantique entre "army" (0.26) et "enemy" (0.24), ce qui confirme mathématiquement la doctrine de Sun Tzu privilégiant le stratagème et la maîtrise du commandement sur l'affrontement brut.

In [69]:
from nltk.collocations import BigramCollocationFinder, TrigramCollocationFinder
from nltk.collocations import BigramAssocMeasures, TrigramAssocMeasures

# On récupère les tokens nettoyés (déjà dans la liste corpus_par_chapitre)
tokens_chap4 = corpus_par_chapitre[3]  # tactical Dispositions
tokens_chap5 = corpus_par_chapitre[4]  # Energy

def extraire_expressions(tokens, n=10):
    #  BIGRAMMES 
    mesures_bi = BigramAssocMeasures()
    chercheur_bi = BigramCollocationFinder.from_words(tokens)
    # On filtre pour ne garder que ce qui apparaît au moins 2 fois
    chercheur_bi.apply_freq_filter(2)
    top_bi = chercheur_bi.nbest(mesures_bi.likelihood_ratio, n)
    
    #  TRIGRAMMES 
    mesures_tri = TrigramAssocMeasures()
    chercheur_tri = TrigramCollocationFinder.from_words(tokens)
    chercheur_tri.apply_freq_filter(2)
    top_tri = chercheur_tri.nbest(mesures_tri.likelihood_ratio, n)
    
    return top_bi, top_tri

# Extraction pour le Chapitre 4
bi4, tri4 = extraire_expressions(tokens_chap4)
# Extraction pour le Chapitre 5
bi5, tri5 = extraire_expressions(tokens_chap5)

In [72]:
print(bi4,tri4)

[('tu', 'mu'), ('estimation', 'quantity'), ('balancing', 'chances'), ('ho', 'shih'), ('defeating', 'enemy'), ('third', 'term'), ('ao', 'kung'), ('chang', 'yu'), ('credit', 'courage'), ('han', 'hsin')] [('tu', 'mu', 'says'), ('ts', 'ao', 'kung'), ('opportunity', 'defeating', 'enemy'), ('li', 'ch', 'uan')]


In [73]:
print(bi5,tri5)

[('tu', 'mu'), ('ang', 'chuan'), ('p', 'ang'), ('chang', 'yu'), ('sun', 'pin'), ('ao', 'kung'), ('ts', 'ao'), ('sun', 'tzŭ'), ('combined', 'energy'), ('indirect', 'tactics')] [('p', 'ang', 'chuan'), ('tu', 'mu', 'says'), ('ts', 'ao', 'kung'), ('without', 'head', 'tail'), ('may', 'without', 'head'), ('order', 'secure', 'victory'), ('first', 'han', 'emperor')]


Chapitre 4 (Tactical Dispositions) :
L'extraction des n-grams du Chapitre 4 révèle une structure textuelle hybride où les références aux commentateurs classiques ("tu mu says", "ao kung") cohabitent avec une terminologie de la probabilité statistique, comme "balancing chances" ou "estimation quantity", soulignant l'aspect analytique et quasi-mathématique de la préparation tactique chez Sun Tzu.

Chapitre 5 (Energy) :
Pour le Chapitre 5, l'émergence de collocations conceptuelles comme "indirect tactics" et "combined energy", ainsi que le trigram "order secure victory", démontre que l'analyse par n-grams parvient à isoler les piliers de la doctrine de Sun Tzu sur la flexibilité opérationnelle et la gestion de la force vive.